# 01 — Correlational analysis: what patterns do we observe?

Notebook 00 created the unit-level backdoor-defense dataset.

The causal question remains:

> **To what extent does `treatment` cause a change in `outcome`?**

with:

- `treatment = 0`: random filtering;
- `treatment = 1`: backdoor defense;
- `outcome = 1`: successful backdoor detection;
- `outcome = 0`: detection failure.

This notebook does **not** estimate that causal effect. It explores associations that will help us ask better questions when constructing the DAG.

### Tutorial path

00 Data preparation → **01 Correlational analysis** → 02 Causal inference


## 1. Configure the exploration

We use:

- **Spearman correlation** for monotonic marginal association;
- Seaborn `vlag` for signed correlations;
- Seaborn `mako` for ordinary numeric plots;
- a reproducible sample for large pairplots.

The `pre_treatment_variables` and `post_treatment_variables` lists record **timing knowledge**. Timing is not learned from correlation; it comes from how the study is defined.


In [ ]:
def default_params():
    return {
        "causal_dataset": "data/causal_data.csv",
        "dag_worksheet_output": "data/dag_worksheet.csv",
        "treatment_column": "treatment",
        "outcome_column": "outcome",
        "covariate_columns": ['code_number_tokens', 'code_complexity', 'code_num_identifiers', 'code_num_strings', 'reviewer_experience', 'rollout_eligibility', 'noise_feature', 'inspection_intensity', 'manual_review_flag'],
        "pre_treatment_variables": ['code_number_tokens', 'code_complexity', 'code_num_identifiers', 'code_num_strings', 'reviewer_experience', 'rollout_eligibility', 'noise_feature'],
        "post_treatment_variables": ['inspection_intensity', 'manual_review_flag'],
        "correlation_method": "spearman",
        "plot_palette": "mako",
        "heatmap_palette": "vlag",
        "top_variables_to_plot": 7,
        "pairplot_max_rows": 3000,
        "random_seed": 42,
    }

params = default_params()
params


## 2. Load the dataset produced by notebook 00

At this stage we use only the observed table. We do not load the hidden synthetic DAG.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.correlational_analysis_utils import (
    build_dag_worksheet,
    correlation_matrix,
    covariate_quality_table,
    load_causal_dataset,
    outcome_by_treatment_table,
    treatment_balance_table,
    variable_association_table,
)

sns.set_theme(style="whitegrid")

analysis_df, columns = load_causal_dataset(params)

treatment = columns.treatment
outcome = columns.outcome
covariates = columns.covariates

print(f"Rows: {len(analysis_df):,}")
print(f"Treatment column: {treatment}")
print(f"Outcome column: {outcome}")
print(f"Covariates: {len(covariates)}")

analysis_df.head()


## 3. Remember what treatment and outcome mean

It is easy to lose the domain meaning when variables are encoded as 0 and 1.

| Value | Interpretation |
|---|---|
| `treatment = 0` | random filtering baseline |
| `treatment = 1` | backdoor defense applied |
| `outcome = 0` | detection failed |
| `outcome = 1` | detection succeeded |

The average of `outcome` within a group is that group's **detection success rate (DSR)**.


## 4. Check whether each covariate varies

A constant variable cannot contribute to a correlation or distinguish treatment groups.

We report any such variable before the main analysis so `NaN` correlations are not mistaken for substantive findings.


In [ ]:
quality_table = covariate_quality_table(
    analysis_df,
    covariates,
)

informative_covariates = (
    quality_table
    .loc[quality_table["has_variation"], "variable"]
    .tolist()
)

constant_covariates = (
    quality_table
    .loc[~quality_table["has_variation"], "variable"]
    .tolist()
)

print(f"Varying covariates: {len(informative_covariates)}")
print(f"Constant covariates: {constant_covariates or 'none'}")

quality_table


## 5. Start with the raw treatment–outcome association

First compare detection success under the two observed treatment groups.

This is a **descriptive difference**, not yet a causal effect. Because treatment assignment is observational, the two groups may differ in other variables that also affect detection success.


In [ ]:
outcome_summary = outcome_by_treatment_table(
    analysis_df,
    treatment=treatment,
    outcome=outcome,
)

outcome_summary["condition"] = outcome_summary[treatment].map(
    {
        0: "Random filtering",
        1: "Backdoor defense",
    }
)

control_dsr = float(
    outcome_summary.loc[outcome_summary[treatment] == 0, "outcome_mean"].iloc[0]
)
defense_dsr = float(
    outcome_summary.loc[outcome_summary[treatment] == 1, "outcome_mean"].iloc[0]
)

print(f"Random-filtering DSR: {control_dsr:.3f}")
print(f"Backdoor-defense DSR: {defense_dsr:.3f}")
print(f"Raw DSR difference:   {defense_dsr - control_dsr:.3f}")

outcome_summary


In [ ]:
plot_df = analysis_df.assign(
    treatment_condition=analysis_df[treatment].map(
        {
            0: "Random filtering",
            1: "Backdoor defense",
        }
    )
)

plt.figure(figsize=(7, 4))

sns.barplot(
    data=plot_df,
    x="treatment_condition",
    y=outcome,
    hue="treatment_condition",
    palette=params["plot_palette"],
    errorbar=("ci", 95),
    legend=False,
)

plt.title("Observed detection success by treatment condition")
plt.xlabel("")
plt.ylabel("Detection success rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


### Why the raw difference is not enough

Suppose more complex code is both:

1. more likely to receive the backdoor defense, and
2. harder to detect successfully.

Then the observed difference between treatment groups mixes the effect of the defense with differences in code complexity.

That is why we need a causal graph rather than a simple group comparison.


## 6. Explore pairwise correlations

The heatmap summarizes marginal association among the observed variables.

Use it to notice:

- variables related to treatment;
- variables related to outcome;
- clusters of strongly related measurements;
- possible proxy or redundant variables.

**Do not infer arrow direction from the heatmap.**


In [ ]:
analysis_columns = [
    treatment,
    outcome,
    *informative_covariates,
]

corr = correlation_matrix(
    analysis_df,
    columns=analysis_columns,
    method=params["correlation_method"],
)

corr.round(2)


In [ ]:
mask = np.triu(
    np.ones_like(corr, dtype=bool),
    k=1,
)

plt.figure(figsize=(12, 9))

sns.heatmap(
    corr,
    mask=mask,
    cmap=params["heatmap_palette"],
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={
        "label": f"{params['correlation_method'].title()} correlation"
    },
)

plt.title("Pairwise associations among observed variables")
plt.tight_layout()
plt.show()


## 7. Which variables differ between treatment groups?

The **standardized mean difference (SMD)** expresses the treated-versus-control difference in standard-deviation units.

- near 0: the groups look similar on that variable;
- farther from 0: the groups are more imbalanced.

Imbalance is useful evidence about treatment assignment, but it is **not proof of confounding**. A post-treatment variable can also be highly imbalanced.


In [ ]:
balance_table = treatment_balance_table(
    analysis_df,
    treatment=treatment,
    covariates=covariates,
)

balance_table.round(3)


In [ ]:
balance_plot = (
    balance_table
    .loc[balance_table["has_variation"]]
    .sort_values("standardized_mean_difference")
)

plt.figure(figsize=(9, 6))

sns.barplot(
    data=balance_plot,
    x="standardized_mean_difference",
    y="variable",
    hue="variable",
    palette=params["plot_palette"],
    legend=False,
)

plt.axvline(0, color="black", linewidth=1)
plt.axvline(-0.10, color="gray", linestyle="--", linewidth=1)
plt.axvline(0.10, color="gray", linestyle="--", linewidth=1)

plt.title("Covariate imbalance: backdoor defense vs random filtering")
plt.xlabel("Standardized mean difference")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 8. Map association with treatment against association with outcome

This plot helps prioritize variables for causal discussion.

A variable associated with both treatment and outcome deserves attention, but the same statistical pattern can be produced by very different causal structures.

For example, a variable might be:

- a pre-treatment common cause;
- a post-treatment mediator;
- a collider;
- a proxy for another variable.

Correlation alone cannot distinguish these possibilities.


In [ ]:
association_table = variable_association_table(
    analysis_df,
    treatment=treatment,
    outcome=outcome,
    covariates=informative_covariates,
    method=params["correlation_method"],
)

association_table.round(3)


In [ ]:
plt.figure(figsize=(9, 7))

ax = sns.scatterplot(
    data=association_table,
    x="association_with_treatment",
    y="association_with_outcome",
    size="screening_score",
    hue="screening_score",
    palette=params["plot_palette"],
    sizes=(70, 320),
    legend=False,
)

plt.axvline(0, color="gray", linewidth=1)
plt.axhline(0, color="gray", linewidth=1)

for row in association_table.itertuples():
    ax.text(
        row.association_with_treatment + 0.008,
        row.association_with_outcome + 0.008,
        row.variable,
        fontsize=9,
    )

plt.title("Observed association with treatment and detection outcome")
plt.xlabel(f"Association with treatment ({params['correlation_method']})")
plt.ylabel(f"Association with outcome ({params['correlation_method']})")
plt.tight_layout()
plt.show()


## 9. Add temporal knowledge before assigning causal roles

This is a crucial step.

Correlation is estimated from the data. **Timing comes from the study design.**

### Known before treatment

The code features, reviewer experience, rollout eligibility, and noise feature are defined before the treatment decision.

### Measured after treatment

`inspection_intensity` and `manual_review_flag` occur after the treatment has been assigned.

A post-treatment variable should not be treated as an ordinary baseline confounder just because it correlates with both treatment and outcome.


In [ ]:
timing_map = {
    **{
        variable: "before treatment"
        for variable in params["pre_treatment_variables"]
    },
    **{
        variable: "after treatment"
        for variable in params["post_treatment_variables"]
    },
}

pd.DataFrame(
    {
        "variable": covariates,
        "measurement_timing": [timing_map[v] for v in covariates],
    }
)


## 10. Inspect a few strong relationships more closely

A correlation coefficient can hide nonlinear structure, clusters, and overlap between treatment groups.

For readability, we plot only a few high-priority variables and sample at most `pairplot_max_rows` observations.


In [ ]:
top_n = min(
    int(params["top_variables_to_plot"]),
    len(association_table),
)

top_variables = association_table.head(top_n)["variable"].tolist()
pairplot_variables = top_variables[: min(4, len(top_variables))]
pairplot_n = min(int(params["pairplot_max_rows"]), len(analysis_df))

if pairplot_variables:
    pairplot_df = (
        analysis_df[[treatment, *pairplot_variables]]
        .sample(
            n=pairplot_n,
            random_state=params["random_seed"],
        )
    )

    sns.pairplot(
        pairplot_df,
        vars=pairplot_variables,
        hue=treatment,
        palette=sns.color_palette("colorblind", n_colors=2),
        corner=True,
        diag_kind="hist",
        plot_kws={
            "alpha": 0.50,
            "s": 22,
        },
    )
    plt.show()
else:
    print("No varying covariates available for the pairplot.")


## 11. Build the DAG worksheet

The worksheet is the handoff to notebook 02.

It combines:

- observed association with treatment;
- observed association with outcome;
- treatment-group imbalance;
- known measurement timing;
- blank fields for your causal reasoning.

Fill the final columns using a mechanism and temporal ordering—not by choosing the variables with the largest correlations.


In [ ]:
dag_worksheet = build_dag_worksheet(
    association_table=association_table,
    balance_table=balance_table,
    quality_table=quality_table,
    treatment=treatment,
    outcome=outcome,
    timing_map=timing_map,
)

dag_worksheet


## 12. Questions to answer before notebook 02

For each variable, ask:

1. Was it determined before or after treatment?
2. Could it cause whether the backdoor defense is applied?
3. Could it cause detection success?
4. Could the treatment itself cause it?
5. Could the outcome cause it?
6. Could it mainly be a proxy for another variable?
7. Would adjusting for it block part of the treatment effect?
8. Could adjusting for it open an unwanted path?
9. What mechanism justifies each proposed arrow?

The point is not to guess the hidden DAG perfectly. The point is to make the assumptions explicit.


In [ ]:
from pathlib import Path

worksheet_path = Path(params["dag_worksheet_output"])
worksheet_path.parent.mkdir(parents=True, exist_ok=True)
dag_worksheet.to_csv(worksheet_path, index=False)

print(f"Saved DAG worksheet to: {worksheet_path}")


## Transition to notebook 02

Notebook 01 answers:

> **What patterns are visible in the observed data?**

Notebook 02 asks:

> **What causal structure do we believe could have generated those patterns, and what does that structure imply about the effect of treatment on outcome?**

Bring the DAG worksheet with you. Do not reveal the hidden synthetic DAG yet.
